In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

seed = 3888
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

img_width, img_height = 50, 50
batch_size = 512
AUTOTUNE = tf.data.AUTOTUNE

tumor_folders = ["Invasive_Tumor", "DCIS_1", "DCIS_2", "Prolif_Invasive_Tumor"]
immune_folders = ["CD4+_T_Cells", "CD8+_T_Cells", "B_Cells", "Stromal"]

for folder in tumor_folders + immune_folders:
    files = tf.io.gfile.glob(os.path.join(folder, "*.png"))
    print(f"{folder}: {len(files)} image(s) found")

clip_limit = 3.0
tile_grid_size = (8, 8)

def apply_clahe(img_np):
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    cl = clahe.apply(l)

    merged = cv2.merge((cl, a, b))
    final_img = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)
    return final_img

def get_file_label_pairs(folders, main_label):
    pairs = []
    for folder in folders:
        files = tf.io.gfile.glob(os.path.join(folder, "*.png"))
        for f in files:
            sub_label = os.path.basename(folder)
            pairs.append((f, main_label, sub_label))
    return pairs

tumor_files = get_file_label_pairs(tumor_folders, main_label=1)
immune_files = get_file_label_pairs(immune_folders, main_label=0)
all_file_label_pairs = tumor_files + immune_files

def create_balanced_equal_dataset(file_label_pairs, target_size=15000):
    from collections import defaultdict
    label_to_files = defaultdict(list)
    for f, main_label, sub_label in file_label_pairs:
        label_to_files[sub_label].append((f, main_label, sub_label))

    balanced = []
    for sub_label, files in label_to_files.items():
        originals = files.copy()
        current_size = len(originals)

        if current_size < target_size:
            extra_needed = target_size - current_size
            oversampled = random.choices(originals, k=extra_needed)
            balanced.extend((f, m, s, False) for f, m, s in originals)
            balanced.extend((f, m, s, True) for f, m, s in oversampled)
        elif current_size > target_size:
            undersampled = random.sample(originals, k=target_size)
            balanced.extend((f, m, s, False) for f, m, s in undersampled)
        else:
            balanced.extend((f, m, s, False) for f, m, s in originals)

    random.shuffle(balanced)
    return balanced

balanced_pairs = create_balanced_equal_dataset(all_file_label_pairs, target_size=15000)

# Makes labels equal to 5 different types
def extract_labels_from_balanced_pairs(pairs):
    file_paths = []
    labels = []
    tumor_map = {
        "Invasive_Tumor": 1,
        "DCIS_1": 2,
        "DCIS_2": 3,
        "Prolif_Invasive_Tumor": 4
    }
    for f, main_label, sub_label, _ in pairs:
        label = 0 if sub_label not in tumor_map else tumor_map[sub_label]
        file_paths.append(f)
        labels.append(label)
    return np.array(file_paths), np.array(labels)

file_paths, labels = extract_labels_from_balanced_pairs(balanced_pairs)

# Does the undersampling
prolif_count = np.sum(labels == 4)
balanced_indices = []
for label in np.unique(labels):
    label_indices = np.where(labels == label)[0]
    if len(label_indices) > prolif_count:
        selected = np.random.choice(label_indices, prolif_count, replace=False)
    else:
        selected = label_indices
    balanced_indices.extend(selected)

file_paths = file_paths[balanced_indices]
labels = labels[balanced_indices]

shuffle_idx = np.random.permutation(len(file_paths))
file_paths = file_paths[shuffle_idx]
labels = labels[shuffle_idx]

train_files, test_files, train_labels, test_labels = train_test_split(
    file_paths, labels, test_size=0.2, stratify=labels, random_state=seed)

# Sam methods again

def decode_img(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [img_width, img_height])
    img = img / 255.0
    return img

def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [img_width, img_height])

    def clahe_wrapper(img_np):
        img_np = img_np.numpy().astype(np.uint8)
        img_np = apply_clahe(img_np)
        return img_np

    img = tf.py_function(
        func=clahe_wrapper,
        inp=[img],
        Tout=tf.uint8
    )
    img.set_shape([img_width, img_height, 3])

    img = tf.cast(img, tf.float32) / 255.0

    return img, label


def prepare_dataset(files, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    ds = ds.map(process_path, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_ds = prepare_dataset(train_files, train_labels, shuffle=True)
test_ds = prepare_dataset(test_files, test_labels)

# Defines my model
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(img_width, img_height, 3)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(5, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_ds, epochs=20, validation_data=test_ds)

# Mark additions

# All for creation of the accuracy-loss plot
hist_df = pd.DataFrame(history.history)
hist_df['epoch'] = hist_df.index + 1

hist_long = pd.melt(
    hist_df,
    id_vars='epoch',
    value_vars=['loss', 'val_loss', 'accuracy', 'val_accuracy'],
    var_name='metric',
    value_name='value'
)

hist_long['data'] = hist_long['metric'].apply(lambda x: 'training' if 'val' not in x else 'validation')
hist_long['type'] = hist_long['metric'].apply(lambda x: 'loss' if 'loss' in x else 'accuracy')

sns.set(style="whitegrid")

g = sns.relplot(
    data=hist_long,
    x="epoch", y="value",
    col="type", hue="data",
    kind="scatter", facet_kws={'sharey': False},
    palette="Set1", height=4, aspect=1.5
)

for ax, metric in zip(g.axes.flat, ['loss', 'accuracy']):
    sns.lineplot(
        data=hist_long[hist_long['type'] == metric],
        x="epoch", y="value", hue="data",
        ax=ax, legend=False
    )

g.set_titles("{col_name}")
g.set_axis_labels("epoch", "value")
plt.tight_layout()
plt.show()

# Evaluation
y_pred = model.predict(test_ds)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.concatenate([y for _, y in test_ds], axis=0)

target_names = ["Immune_Cells", "Invasive_Tumor", "DCIS_1", "DCIS_2", "Prolif_Invasive_Tumor"]
print(classification_report(y_true, y_pred_classes, target_names=target_names))

cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

Invasive_Tumor: 0 image(s) found
DCIS_1: 0 image(s) found
DCIS_2: 0 image(s) found
Prolif_Invasive_Tumor: 0 image(s) found
CD4+_T_Cells: 0 image(s) found
CD8+_T_Cells: 0 image(s) found
B_Cells: 0 image(s) found
Stromal: 0 image(s) found


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.